# NYC Taxi Trips — Data Visualization & Business Intelligence
**Course:** BDML2 · **Academic year:** 2026 · **Submission:** `DataViz_20231654_KARAKOZIAN2026BDML2.ipynb`

**Research question:** *When, where, and how do people use NYC taxis, and what operational insights can the data support?*

**Source:** [Kepler.gl NYC Taxi sample](https://github.com/uber-web/kepler.gl-data). This is a **sample**, not a census of every NYC taxi trip. Conclusions describe this sample only.

**Notebook workflow:** Load → audit → clean → create features → visualize → interactive mapping → KPIs → evidence-based storytelling.

> Run cells in order. Prepared for student 20231654 — KARAKOZIAN. Charts and numerical conclusions are generated only after loading the dataset.

## 0 — Environment and output folders

Create and activate `.venv`, then run `pip install -r requirements.txt` **in your terminal**. In VS Code, select the `.venv` interpreter as the notebook kernel. This notebook does not install dependencies.

When you run the imports cell, an `images/` folder is created automatically in the notebook's working directory. Static Matplotlib figures are exported to PNG and interactive Plotly figures to HTML. Plotly PNG exports are attempted when `kaleido` is installed and configured; the notebook still works without it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import plotly.express as px
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Libraries imported successfully.")
from pathlib import Path

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Export folder: {IMAGES_DIR.resolve()}")

def save_plotly(fig, stem):
    """Always save interactive HTML; try a PNG preview without blocking the notebook."""
    html_path = IMAGES_DIR / f"{stem}.html"
    fig.write_html(str(html_path), include_plotlyjs=True)
    print(f"Saved interactive chart: {html_path}")
    try:
        fig.write_image(str(IMAGES_DIR / f"{stem}.png"), width=1150, height=720, scale=1.5)
        print(f"Saved PNG: {IMAGES_DIR / (stem + '.png')}")
    except (ImportError, ValueError, RuntimeError, OSError) as exc:
        print("PNG export skipped (Plotly Kaleido/Chrome configuration missing). HTML export is available.")


## 1 — Load and inspect the dataset
This is a recorded sample of NYC taxi journeys. First inspect actual columns and types before transforming anything.

In [ ]:
path2 = "https://raw.githubusercontent.com/uber-web/kepler.gl-data/master/nyctrips/data.csv"
df2 = pd.read_csv(path2, delimiter=",")
print(f"Raw observations: {len(df2):,} | Columns: {df2.shape[1]}")
display(df2.head())
print("Columns:", df2.columns.tolist())
df2.info()

In [ ]:
display(df2.describe(include="all").T)
missing = pd.DataFrame({
    "missing_count": df2.isna().sum(),
    "missing_percent": (df2.isna().mean() * 100).round(2),
}).sort_values("missing_percent", ascending=False)
display(missing)

### Missingness — Missingno
A missingness matrix reveals both gaps and their patterns; it does not itself tell us why a field is missing.

In [ ]:
msno.matrix(df2, figsize=(13, 4), sparkline=False)
plt.title("Missing values in raw taxi records")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "missing_values_matrix.png", dpi=180, bbox_inches="tight")
plt.show()
msno.bar(df2, figsize=(13, 4))
plt.title("Non-missing observations by field")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "missing_values_bar.png", dpi=180, bbox_inches="tight")
plt.show()

## 2 — Prepare features and clean invalid records
**Decisions:** parse datetimes with `utc=True` to handle offset-tagged strings consistently; calculate duration in minutes; extract the **clock hour printed in the original pickup timestamp** so we do not silently interpret the source timezone as confirmed NYC local time. Remove invalid date/distance/duration/location records. The upper distance/duration thresholds are practical quality filters, not claims that longer trips never occur.

In [ ]:
df = df2.copy()
required = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
    "pickup_latitude", "pickup_longitude",
]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset columns changed; missing: {missing_columns}")

# Preserve the recorded wall-clock time for the hourly visualization.
# For strings such as '2015-01-15 19:05:39 +00:00', this yields 19.
pickup_clock = pd.to_datetime(df["tpep_pickup_datetime"].astype("string").str.slice(0, 19), errors="coerce")
df["hour"] = pickup_clock.dt.hour
df["day_of_week"] = pickup_clock.dt.day_name()
df["date"] = pickup_clock.dt.date

df["pickup_dt"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce", utc=True)
df["dropoff_dt"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce", utc=True)
df["duration_min"] = (df["dropoff_dt"] - df["pickup_dt"]).dt.total_seconds() / 60

for col in ["trip_distance", "pickup_latitude", "pickup_longitude", "duration_min"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Bounding box for NYC area (approximate), not a borough polygon.
valid_mask = (
    df["pickup_dt"].notna() & df["dropoff_dt"].notna() &
    df["hour"].notna() &
    df["trip_distance"].between(0.01, 100) &
    df["duration_min"].between(0.5, 240) &
    df["pickup_latitude"].between(40.45, 40.95) &
    df["pickup_longitude"].between(-74.30, -73.65)
)
raw_count = len(df)
df = df.loc[valid_mask].copy()
df["speed_mph"] = df["trip_distance"] / (df["duration_min"] / 60)
# Flag implausible average journey speeds as additional data quality checks.
df = df.loc[df["speed_mph"].between(0.1, 80)].copy()

display(pd.DataFrame({"status": ["raw", "clean", "excluded"],
                      "records": [raw_count, len(df), raw_count-len(df)]}))
print(f"Retained {len(df)/raw_count:.1%} of records")
if df.empty:
    raise ValueError("No trips remain: inspect source columns and filtering rules.")
display(df[["trip_distance", "duration_min", "hour", "speed_mph"]].describe().T)

### Data quality note
*We use trip distances in miles (as defined in the taxi data), minutes for trip duration, and average miles/hour for implied journey speed. Speed is derived from aggregate trip distance and elapsed time; it is **not** an instantaneous traffic measurement. The cleaning filters can bias results against extremely long or fast trips.*

## 3 — KPI dashboard
Core indicators designed for a taxi dispatch / demand planning audience.

In [ ]:
kpis = pd.DataFrame({
    "KPI": ["Valid trips", "Average trip distance", "Median trip distance",
            "Average trip duration", "Median trip duration", "Median implied speed"],
    "Value": [f"{len(df):,}", f"{df.trip_distance.mean():.2f} miles",
              f"{df.trip_distance.median():.2f} miles",
              f"{df.duration_min.mean():.1f} min",
              f"{df.duration_min.median():.1f} min",
              f"{df.speed_mph.median():.1f} mph"]
})
display(kpis.style.hide(axis="index"))

## 4 — Statistical visualizations

### 4.1 — Distribution of journey distance
**Question:** Are most trips short or long? We show the full descriptive statistics, while clipping the **plot only** at the 99th percentile to preserve legibility.

In [ ]:
p99 = df["trip_distance"].quantile(.99)
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df.loc[df["trip_distance"] <= p99, "trip_distance"], bins=45, ax=ax)
ax.axvline(df["trip_distance"].median(), color="crimson", ls="--", label="Median (all valid trips)")
ax.set(xlabel="Trip distance (miles)", ylabel="Number of trips", title="Taxi trip distances (displayed up to 99th percentile)")
ax.legend(); plt.tight_layout()
plt.savefig(IMAGES_DIR / "trip_distances.png", dpi=220, bbox_inches="tight")
plt.show()
print(f"Median: {df.trip_distance.median():.2f} miles; 90th percentile: {df.trip_distance.quantile(.90):.2f} miles")

### 4.2 — Hourly demand
**Question:** At what hours are pickups most frequent? Interpret this as the dataset’s recorded clock hours (timezone semantics are not independently verified).

In [ ]:
hourly = df.groupby("hour").size().reindex(range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(x=hourly.index, y=hourly.values, ax=ax)
ax.set(xlabel="Recorded pickup hour", ylabel="Trips", title="Number of taxi pickups by hour")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "hourly_demand.png", dpi=220, bbox_inches="tight")
plt.show()
print(f"Peak hour: {hourly.idxmax():02d}:00–{(hourly.idxmax()+1)%24:02d}:00 with {hourly.max():,} trips")
print("Recorded dates:", df["date"].nunique())

### 4.3 — Trips by day of the week
**Question:** Is pickup activity evenly distributed across the weekdays represented in this sample? Interpret with caution if only a few dates are present.

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_counts = df["day_of_week"].value_counts().reindex(weekday_order, fill_value=0)
fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(x=weekday_counts.index, y=weekday_counts.values, ax=ax)
ax.set(xlabel="Day of week", ylabel="Trips", title="Taxi pickups by weekday in the recorded sample")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "weekday_demand.png", dpi=220, bbox_inches="tight")
plt.show()

### 4.4 — Duration versus distance (interactive)
**Question:** Do long-distance trips also take longer? Hover to inspect individual sampled records. A relationship does **not** by itself prove congestion.

In [ ]:
sample_scatter = df.sample(n=min(5000, len(df)), random_state=42)
fig = px.scatter(sample_scatter, x="trip_distance", y="duration_min", color="hour",
                 opacity=.55,
                 hover_data=["hour", "day_of_week", "speed_mph"],
                 labels={"trip_distance": "Trip distance (miles)",
                         "duration_min": "Duration (minutes)", "hour": "Pickup hour"},
                 title="Trip duration versus trip distance — sampled rides")
fig.update_layout(height=520)
save_plotly(fig, "duration_vs_distance")
fig.show()

## 5 — Geographic visualization of pickup locations
**Question:** Which areas contain clusters of pickups? For performance the map uses a reproducible subset of up to 4,000 trips. Spatial density here is *sample pickup density*, not normalized demand per resident or per square kilometer.

In [ ]:
map_df = df.loc[
    df["pickup_latitude"].between(40.4, 41.0) &
    df["pickup_longitude"].between(-74.3, -73.6)
].sample(n=min(4000, len(df)), random_state=42).copy()
map_df["pickup_hour"] = map_df["hour"].astype(int).astype(str).str.zfill(2) + ":00"
map_args = dict(
    data_frame=map_df,
    lat="pickup_latitude", lon="pickup_longitude", color="hour",
    zoom=9.5, height=610, opacity=.6,
    hover_data={"pickup_hour": True, "trip_distance": ":.2f", "duration_min": ":.1f",
                "pickup_latitude": False, "pickup_longitude": False},
    title="NYC taxi pickup locations (sample)",
)
# CARTO map tiles worked in the user's environment; fall back for older Plotly versions.
if hasattr(px, "scatter_map"):
    fig = px.scatter_map(**map_args, map_style="carto-positron")
    fig.update_layout(map=dict(center=dict(lat=40.73, lon=-73.95)))
else:
    fig = px.scatter_mapbox(**map_args, mapbox_style="carto-positron")
    fig.update_layout(mapbox=dict(center=dict(lat=40.73, lon=-73.95)))
fig.update_layout(margin={"l": 0, "r": 0, "t": 55, "b": 0})
save_plotly(fig, "pickup_map")
fig.show()

## 6 — Additional BI analysis: demand share & possible congestion
Average implied speed by recorded pickup hour can **suggest** differences in traffic conditions, but does not isolate congestion from route choices, pickup behaviors, or data quality. Use medians to reduce outlier sensitivity.

In [ ]:
hourly_bi = df.groupby("hour").agg(
    trips=("trip_distance", "size"),
    median_duration=("duration_min", "median"),
    median_speed_mph=("speed_mph", "median"),
    median_distance=("trip_distance", "median")
).reindex(range(24))
hourly_bi["share_percent"] = hourly_bi["trips"] / len(df) * 100
display(hourly_bi.round(2))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hourly_bi.index, hourly_bi.median_speed_mph, marker="o")
ax.set(xlabel="Recorded pickup hour", ylabel="Median implied speed (mph)",
       title="Median implied trip speed by pickup hour")
ax.set_xticks(range(24)); plt.tight_layout()
plt.savefig(IMAGES_DIR / "median_speed_by_hour.png", dpi=220, bbox_inches="tight")
plt.show()

## 7 — Evidence-based storytelling (auto-generated)
The text below computes the findings **from the data**. Read it and edit the narrative to reflect what you actually see in the map. Do not claim an airport hotspot, commuter behavior, or causal congestion without additional geographic/time evidence.

In [ ]:
peak = int(hourly.idxmax())
short_share = 100 * df["trip_distance"].le(2).mean()
correlation = df[["trip_distance", "duration_min"]].corr().iloc[0, 1]
low_speed_hour = int(hourly_bi["median_speed_mph"].idxmin())
report = f"""
### Executive summary — NYC Taxi Trips

- **Dataset and coverage:** {len(df):,} valid trips out of {raw_count:,} initial records; {df['date'].nunique()} distinct recorded date(s). These findings relate to the **sample**, not all NYC travel.
- **Short-ride market:** {short_share:.1f}% of cleaned journeys cover at most **2 miles**. The median journey is {df.trip_distance.median():.2f} miles.
- **Time-of-day demand:** the busiest recorded pickup hour is **{peak:02d}:00** with **{hourly.max():,} trips** ({100*hourly.max()/len(df):.1f}% of the cleaned sample). Check for additional peaks in the bar chart before attributing any pattern to commuting.
- **Distance and duration:** Pearson correlation is **{correlation:.2f}** within the cleaned data. Individual rides differ in duration even at similar distances; possible reasons include routes, stops and traffic, but these are not directly measured.
- **Speed proxy:** the lowest hourly median implied speed is observed at **{low_speed_hour:02d}:00**; this should not be presented as proof of congestion.
- **Spatial pattern:** visually inspect the map for clusters; identify neighborhoods only if the plotted points support them. Airport hotspots require additional checks or geographic definitions.

### Business implications
A dispatch planner could compare hourly peaks with available driver supply, then examine dense pickup zones. However, sample selection and limited date coverage prevent generalizing precise citywide demand forecasts.

### Limitations
This is a sample dataset; observations are not a randomized representative survey. Dates and recorded timezone labels need confirmation. Trip coordinates can be inaccurate. Cleaning thresholds exclude unusual rides. Geographic counts are not normalized by local population or area.
"""
display(Markdown(report))

## 8 — Final personal reflection (edit before submission)
In 4–6 sentences, describe **which graph surprised you**, **one map hotspot you directly observed**, **one business action**, and **one important limitation**. This personal interpretation is more valuable than unsupported generic claims.

**Suggested wording:** *My analysis reveals ... . The hourly chart suggests ... . On the map I observe ... . This could help ... . However, the dataset is limited because ... .*

## 9 — Export inventory and GitHub submission

All saved files appear under `images/`. PNG files can be embedded directly in your `README.md`, for example `![Hourly pickups](images/hourly_demand.png)`. Interactive Plotly figures are saved as self-contained HTML files you can open in your browser. PNG snapshots of Plotly figures require optional `kaleido` support (and, in recent versions, Chrome).

For GitHub, commit the notebook, `README.md`, `requirements.txt`, `.gitignore`, and selected exports. Do **not** commit `.venv/`. Run all cells, review the results, and replace the personal reflection with your own interpretation.

In [ ]:
print(f"Export directory: {IMAGES_DIR.resolve()}")
for path in sorted(IMAGES_DIR.iterdir()):
    if path.is_file():
        print(f"- {path.name}")